# bert-1
- run pipeline

In [ ]:
from nlp import analyze_reports

stats = analyze_reports('../data/reports/Baosteel')
# stats = analyze_reports("../data/reports")

# bert-2
- run vizualisations

In [ ]:
from nlp import visualize_results

visualize_results("../cache", "../out")

✅ Loaded: 193 reports, 15 companies, 2013-2024

EXPORTING CSV FILES
   ✓ company_year.csv (129 rows)
   ✓ company_totals.csv (15 companies)
   ✓ yearly_industry.csv (12 years)
   ✓ funnel_company_year.csv (129 rows)

   📁 All CSVs saved to: ../out/

GENERATING PLOTS
   ✓ slide_main.png
   ✓ slide_sentiment_trend.png
   ✓ talk_score_trend.png
   ✓ funnel_trend.png
   ✓ talk_score_per_company.png
   ✓ per_company_components.png
   ✓ per_company_sentiment.png
   ✓ sentiment_all_companies.png
   ✓ n0_funnel.png
   ✓ n0_quality_comparison.png
   ✓ n0_per_company.png
   ✓ n0_gap_analysis.png

   📁 All plots saved to: ../out/

   Generating word frequency plots...

   📊 ALL CHUNKS (top 30):
   environment(17950), sustainable(17493), production(16608), emission(14006), management(13797), energy(12048), financial(11688), development(11410), business(11035), reduction(10864), products(9790), million(9437), carbon(9152), iron(8384), process(6902)

   🌱 OPPORTUNITY chunks (top 15):
   production(7

# rag

- Model Testing: Test which models work for extraction (format compliance + quality).

In [ ]:
from nlp.test import save_test_results, test_models, compare_extractions
from nlp import Config

# Available models (for reference):
# Groq:   mixtral-8x7b-32768, gemma2-9b-it
# Ollama: gemma3:4b

MODELS = [

    # RAG approach
    Config(llm_provider="groq", model="llama-3.1-8b-instant", approach="rag", ctx=128000, top_k=20, reuse_faiss_cache=False),

    # Exthaustive approach
    # Config(llm_provider="groq", model="llama-3.1-8b-instant", approach="exhaustive", ctx=128000, batch_size=3),  # free tier: 6k TPM - test batch_size
    # Config(llm_provider="ollama", model="qwen2.5:3b", approach="exhaustive", ctx=4096),  # 1.9GB - FAIL format
    # Config(llm_provider="ollama", model="qwen3:4b", approach="exhaustive", ctx=2048),     # 2.5GB - too slow. was generating alot of hidden thinking tokens
    # Config(llm_provider="ollama", model="gemma3:4b", approach="exhaustive", ctx=4096),   # 3.3GB - PASS but slow
    # Config(llm_provider="ollama", model="llama3.2:3b", approach="exhaustive", ctx=4096),  # too small for quality
    # Config(llm_provider="ollama", model="llama3.1:8b ", approach="exhaustive", ctx=4096),  # too big for local VRAM

]

results = test_models(MODELS, skip_extraction=False)
compare_extractions(results)
save_test_results(results)

- setup rag pipeline

In [ ]:
from nlp import load_pipeline, Config

config = Config(llm_provider="groq", model="llama-3.1-8b-instant", approach="rag", ctx=128000, top_k=20, reuse_faiss_cache=True)
# Exhaustive (all chunks batched through LLM)
# config = Config(llm_provider="groq", model="llama-3.1-8b-instant", approach="exhaustive", ctx=128000, batch_size=3)
# config = Config(llm_provider="ollama", model="llama3.2:3b", approach="exhaustive", ctx=4096)

pipeline = load_pipeline(config)
pipeline.print_chunk_overview()

- run it

In [4]:
results = pipeline.extract_all_companies(resume=True)


EXTRACTION RUN
  LLM: groq/llama-3.1-8b-instant
  Context: 128,000 tokens → 72 chunks/batch
  Chunks: 15593 (1750 avg chars, 38-19629 range)
  Groups: 132 | Est. LLM calls: 568
  Resuming: 13 companies skipped (CSVs exist), 2 remaining


Extracting: AcciaieriedItalia (009)
Years: ['2021', '2022']
Retrieval: top_k=20, strategy=mmr


  009:   0%|          | 0/2 [00:00<?, ?it/s]

Loading Groq: llama-3.1-8b-instant


    B done: 5 found in 1 batches (0.5s)          
  ✓ Extracted 5 barriers, 0 motivators

Extracting: TataSteelUK (010)
Years: ['2021', '2022', '2023', '2024']
Retrieval: top_k=20, strategy=mmr


  ✓ Extracted 0 barriers, 0 motivators

✓ EXTRACTION COMPLETE
  Time: 1.0s (0.0min)
  Results: 1698 barriers, 1255 motivators
  Resumed: 13 companies from cache
  Saved: ../out/stats.json



# topic modelling

## Grid Search

Staged tuning — each stage narrows the search space for the next.
- **Stage 1**: Embedding model (manual, ~30s encode each — pick 2-3 candidates)
- **Stage 2**: HDBSCAN structure (eom/leaf x min_cluster_size — the biggest knob)
- **Stage 3**: UMAP geometry (n_components x n_neighbors — secondary but real impact)

In [ ]:
from nlp import TopicGridSearch
gs = TopicGridSearch(data_folder="../out", output_folder="../out/topics")

s1 = gs.stage1_embeddings([
    # Strong clustering (MTEB clustering score in parens)
    # (model, batch_size[, dtype]) — batch_size affects speed/size, not embedding values

    ("KaLM-Embedding/KaLM-embedding-multilingual-mini-instruct-v2.5", 32),  # (59.3) 1.9GB
    ("Tarka-AIR/Tarka-Embedding-150M-V1", 64), # (56.4) surprisingly strong, 576MB
    ("ibm-granite/granite-embedding-english-r2", 64),  # (50.8) English-only, 284MB >> WINNER:)
    ("BAAI/bge-small-en-v1.5",          64),   # (39.9) fast, 127MB
    ("Snowflake/snowflake-arctic-embed-s", 64), # (33.8) high aggregate MTEB rank but weak on clustering
    ("sentence-transformers/all-mpnet-base-v2", 64),   # (31.9) STS model, weak for clustering

    # Dropped — model weights alone fill the 3.63GB GPU
    # ("Qwen/Qwen3-Embedding-0.6B",       16),   # (52.3) decoder-only. no fit.
    # ("codefuse-ai/F2LLM-0.6B",16),    # not fitting to design test steps.
    # ("codefuse-ai/F2LLM-1.7B", 4),  # (68.5) OOM even with bfloat16 — MTEB 3.2GB already reflects bfloat16 weights (1.72B×2bytes); ~3.47GB on GPU leaves no headroom
])

In [ ]:
gs.locked["embedding_model"] = "ibm-granite/granite-embedding-english-r2"
param_grid = {
    "hdbscan_min_cluster_size": [8, 12, 16, 20, 25, 30, 40],
    "hdbscan_cluster_selection_method": ["eom", "leaf"],
    "hdbscan_min_samples": [2, 3, 5],
}
s2 = gs.stage2_hdbscan(param_grid=param_grid)

In [ ]:
# Override if needed:
# gs.locked["barriers"] = {"hdbscan_min_cluster_size": 25, "hdbscan_cluster_selection_method": "eom", "hdbscan_min_samples": 5}
# gs.locked["motivators"] = {"hdbscan_min_cluster_size": 16, "hdbscan_cluster_selection_method": "eom", "hdbscan_min_samples": 3}

s3 = gs.stage3_umap()
# Override if needed: gs.locked["barriers"].update({"umap_n_components": 15, "umap_n_neighbors": 25})
print(gs.category_overrides)
# Paste the above dict into TopicModelConfig(category_overrides=...) in the main pipeline cell

In [ ]:
from nlp import TopicModelConfig, run_topic_modeling_pipeline

# Best params from grid search (gs_stage1/2/3)
config = TopicModelConfig(
    # Embedding model
    # embedding_model="ibm-granite/granite-embedding-english-r2",
    embedding_model="KaLM-Embedding/KaLM-embedding-multilingual-mini-instruct-v2.5"
    batch_size=64,

    # -> override base defaults per category (see TopicModelConfig in topic_modelling.py)
    category_overrides={
        "barriers": {
            "hdbscan_min_cluster_size": 25,
            "hdbscan_min_samples": 5,
            "hdbscan_cluster_selection_method": "eom",
            "umap_n_components": 5,
            "umap_n_neighbors": 15,
        },
        "motivators": {
            "hdbscan_min_cluster_size": 16,
            "hdbscan_min_samples": 3,
            "hdbscan_cluster_selection_method": "eom",
            "umap_n_components": 5,
            "umap_n_neighbors": 25,
        },
    },
)

results = run_topic_modeling_pipeline(
    data_folder="../out",
    output_folder="../out/topics",
    config=config,
)
# Access results
barriers_df = results['barriers']['df']
motivators_df = results['motivators']['df']